# 01 -- MLB Schedule Ingestion

Fetch game schedules with confirmed starting pitchers for seasons 2015-2024.

**Requirement:** DATA-01

**Source:** MLB Stats API via `statsapi.schedule()`

**Columns:** game_id, game_date, home_team, away_team, home_probable_pitcher, away_probable_pitcher, home_score, away_score, winning_team, losing_team, status, is_shortened_season, season_games, season

In [ ]:
# If editable install is not set up, uncomment the next line:
# import sys; sys.path.insert(0, "..")

from src.data.mlb_schedule import fetch_schedule
from src.data.cache import load_manifest

In [ ]:
# Fetch schedules for all 10 backtest seasons
seasons = range(2015, 2025)
dfs = {}
for season in seasons:
    print(f"Fetching {season}...")
    dfs[season] = fetch_schedule(season)
    print(f"  {len(dfs[season])} games")

In [ ]:
# Summary: show sample from 2023
df_sample = dfs[2023]
print(f"Columns: {list(df_sample.columns)}")
print(f"\nSample (2023):")
display(df_sample[["game_date", "home_team", "away_team", "home_probable_pitcher", "away_probable_pitcher"]].head(10))

In [ ]:
# Coverage validation
for season, df in dfs.items():
    n_teams = df["home_team"].nunique()
    # 2020 was a 60-game season; other years are 162 games
    expected_games = 60 * 15 if season == 2020 else 162 * 15  # approximate
    flag = "OK" if len(df) > expected_games * 0.9 else "LOW"
    print(f"{season}: {len(df)} games, {n_teams} teams [{flag}]")

In [ ]:
# Cache check
manifest = load_manifest()
schedule_keys = {k: v for k, v in manifest.items() if k.startswith("schedule_")}
print(f"Cached schedule files: {len(schedule_keys)}")
for k, v in sorted(schedule_keys.items()):
    print(f"  {k}: {v['row_count']} rows, fetched {v['fetch_date'][:10]}")